# Benchmarking Datasets

In [1]:
# Get the root path and data paths
#

from pathlib import Path


def repo_root(marker: str = "uv.lock") -> Path:
    """Nearest ancestor of the working directory containing *marker*."""
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / marker).is_file():
            return candidate
    raise FileNotFoundError(f"No {marker} found above {start}")

DATA_IN = repo_root() / "data_in"



In [2]:
# Hugging Face Dataset - BRIGHTER
# NB uses 'joy' not 'happiness' as used by Eckman6

import pandas as pd
from datasets import concatenate_datasets, load_dataset

REPO = "brighter-dataset/BRIGHTER-emotion-categories"

# Extract all the BRIGHTER splits and add source
splits = load_dataset(REPO, "eng")
raw_ds = concatenate_datasets([d.add_column("split", [name] * len(d)) for name, d in splits.items()])
# raw_ds = raw_ds.remove_columns("emotions").rename_column("joy", "happiness")
raw_ds = raw_ds.remove_columns("emotions")
raw_ds = raw_ds.add_column("source", ["brighter-eng"] * len(raw_ds))
raw_ds = raw_ds.select_columns(["source", "split", "id", "text", "anger", "disgust",
                                "fear", "joy", "sadness", "surprise"])

raw_ds.to_parquet(DATA_IN / "brighter_emotions_raw.parquet")

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

1543737

## Use the data

In [6]:
# Get the simple examples and BRIGHTER dataset into dfs
#

simple_df = pd.read_csv(DATA_IN / "simple_text_ekman6.csv")
simple_df["emotion"] = simple_df["emotion"].fillna("none").astype("category")

brighter_df = pd.read_parquet(DATA_IN / "brighter_emotions_raw.parquet")
brighter_df = brighter_df.rename(columns={"joy": "happiness"}, errors="raise")